# True Stellar Obliquity ψ (SOCat)

Compute the **true 3-D spin–orbit angle** ψ from the projected obliquity λ, the orbital inclination i₀, and the stellar inclination i⋆, following Masuda & Winn (2020). i⋆ is derived from v sin i⋆, the rotation period P_rot, and the stellar radius R⋆ via an MCMC.

Powered by a vendored, Pyodide-friendly subset of [coPsi](https://github.com/emilknudstrup/coPsi) (`statsmodels`→`scipy`, no multiprocessing). `emcee` is installed in the browser in the next cell.

In [ ]:
%pip install -q emcee

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from socat import istar   # vendored minimal coPsi.iStar
print('istar loaded')

## Inputs
Give value + 1σ for each. (Only symmetric uncertainties here, like the SOCat web tool.)

In [ ]:
# stellar rotation observables (for i*)
Rs_mu,   Rs_err   = 1.00, 0.05    # R* [Rsun]
Prot_mu, Prot_err = 10.0, 0.20    # P_rot [days]
vsini_mu,vsini_err= 4.0,  0.30    # v sin i* [km/s]

# geometry
lam_mu,  lam_err  = 30.0, 5.0     # projected obliquity λ [deg]
inco_mu, inco_err = 89.0, 0.3     # orbital inclination i_o [deg]

N = 4000          # posterior draws for ψ
ndraws = 5000     # MCMC steps for i*  (raise for smoother posteriors)

## 1. Stellar inclination i⋆ (MCMC)

In [ ]:
s = istar.iStar(Rs=(Rs_mu, Rs_err, 0, 10, 'gauss'),
                Prot=(Prot_mu, Prot_err, 0, 100, 'gauss'),
                vsini=(vsini_mu, vsini_err, 0, 1000, 'gauss'))
res = s.stellarInclination(ndraws=ndraws, nwalkers=100, progress=True)
istar_v, istar_lo, istar_up = res['incs'][0], res['incs'][1], res['incs'][2]
print(f'i* = {istar_v:.1f} +{istar_up:.1f} -{istar_lo:.1f} deg')

## 2. True obliquity ψ

In [ ]:
s.createDistributions(N=N)
# draw the three input distributions
s.dist['incs'] = np.random.normal(istar_v, 0.5*(istar_lo+istar_up), N)
s.dist['inco'] = np.random.normal(inco_mu, inco_err, N)
s.dist['lam']  = np.random.normal(lam_mu,  lam_err,  N)
s.coPsi()

psi = s.dist['psi']
pv, pu, pl = s.getConfidence(psi)
print(f'psi = {pv:.1f} +{pu:.1f} -{pl:.1f} deg')

xk, yk = s.getKDE(psi)
v, up, low = s.getConfidence(psi)
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(xk, yk, 'k')
m = (xk > v-low) & (xk < v+up)
ax.fill_between(xk[m], yk[m], color='C0', alpha=0.5, label='68% HPD')
ax.axvline(v, color='k'); ax.set_ylim(bottom=0)
ax.set_xlabel('ψ  [deg]'); ax.set_ylabel('KDE'); ax.legend()
ax.set_title(f'True obliquity  ψ = {pv:.1f} +{pu:.1f} -{pl:.1f}°')
plt.tight_layout(); plt.show()